<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.5-lora-and-qlora-adapters/notebooks/exercises-7_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.5: LoRA, QLoRA & Adapters

*Module 7 · Fine-Tuning LLMs LIVE CLASS*

> A 7B model has 7 billion parameters. Full fine-tuning updates all of them — needs 4× A100 GPUs. LoRA updates only 0.1% of them via low-rank matrices. QLoRA quantizes the model to 4-bit first, then applies LoRA — fitting a 7B model on a single 16GB GPU . This lesson implements both from scratch, then uses HuggingFace PEFT for production fine-tuning.

`LoRA Math` · `QLoRA 4-bit` · `PEFT` · `HuggingFace` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-7.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — LoRA from Scratch — The Math Made Simple — `01_lora_scratch.py`
2. Step 2 — QLoRA — Quantize First, Then LoRA — `02_qlora.py`
3. Step 3 — HuggingFace PEFT — Production Fine-Tuning — `03_peft_training.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers torch datasets peft bitsandbytes trl accelerate


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · LoRA from Scratch — The Math Made Simple

A low-rank decomposition that adds tiny trainable matrices to frozen weights.

**`01_lora_scratch.py`** · _Block 1 of 3_

LoRA FROM SCRATCH — Understand the math


In [ ]:
# LoRA FROM SCRATCH — Understand the math
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    """Low-Rank Adaptation: W_frozen + B @ A (trainable)."""
    def __init__(self, original_layer, rank=8, alpha=16):
        super().__init__()
        self.original = original_layer
        self.original.weight.requires_grad = False  # FREEZE
        d_in = original_layer.in_features
        d_out = original_layer.out_features
        # Low-rank matrices: d_in → rank → d_out
        self.A = nn.Parameter(torch.randn(d_in, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, d_out))
        self.scale = alpha / rank

    def forward(self, x):
        # Original (frozen) + LoRA (trained)
        original_out = self.original(x)
        lora_out = (x @ self.A @ self.B) * self.scale
        return original_out + lora_out

# ── Demo: Apply LoRA to a linear layer ──
d = 4096  # typical hidden dimension
original = nn.Linear(d, d)
lora = LoRALayer(original, rank=8, alpha=16)

total = sum(p.numel() for p in original.parameters())
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)

print("LoRA from Scratch:\n")
print(f"  Original layer: {d}×{d} = {total:,} params (FROZEN)")
print(f"  LoRA A: {d}×8 = {d*8:,} params")
print(f"  LoRA B: 8×{d} = {8*d:,} params")
print(f"  Trainable: {trainable:,} ({trainable/total*100:.2f}% of original)")
print(f"  Output = W·x + scale * B·A·x")
print(f"\n  For a 7B model with LoRA on attention layers:")
print(f"  ~7B frozen + ~4M trained = 0.06% trainable")
print(f"  Full fine-tune: 4x A100. LoRA: 1x T4/A10.")

# Verify shapes
x = torch.randn(1, d)
out = lora(x)
print(f"\n  Input: {x.shape} → Output: {out.shape} ✓")


> **What just happened?** A 4096×4096 linear layer (16M params) was frozen. Two tiny matrices A (4096×8) and B (8×4096) were added — only 65K trainable params (0.4%). Output = frozen(x) + scale × B(A(x)). The rank r=8 is the key hyperparameter: higher rank = more capacity but more params. r=8 to 32 works for most tasks. This simple trick reduces GPU memory by 10-100x.


### Step 2 · QLoRA — Quantize First, Then LoRA

Load the model in 4-bit precision. Apply LoRA on top. 7B model on 16GB GPU.

**`02_qlora.py`** · _Block 2 of 3_

QLoRA — 4-bit Quantization + LoRA — pip install transformers peft bitsandbytes accelerate


In [ ]:
# QLoRA — 4-bit Quantization + LoRA
# pip install transformers peft bitsandbytes accelerate
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# ── Step 1: Load model in 4-bit ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 (best for LLMs)
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,    # quantize the quantization constants
)

model_name = "google/gemma-2b"  # 2B for demo; use 7B+ in production
# model = AutoModelForCausalLM.from_pretrained(
#     model_name, quantization_config=bnb_config, device_map="auto")
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# ── Step 2: Prepare for LoRA ──
# model = prepare_model_for_kbit_training(model)

# ── Step 3: Apply LoRA ──
lora_config = LoraConfig(
    r=16,                    # rank (8-64, start with 16)
    lora_alpha=32,            # scaling factor (usually 2x rank)
    target_modules=[         # which layers to adapt
        "q_proj", "k_proj",    # attention query & key
        "v_proj", "o_proj",    # attention value & output
    ],
    lora_dropout=0.05,       # regularization
    bias="none",              # don't train biases
    task_type="CAUSAL_LM",
)

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

print("QLoRA Configuration:\n")
print(f"  Quantization: 4-bit NormalFloat (NF4)")
print(f"  Double quant: True (saves additional memory)")
print(f"  Compute dtype: bfloat16\n")
print(f"  LoRA rank: {lora_config.r}")
print(f"  LoRA alpha: {lora_config.lora_alpha}")
print(f"  Target: {lora_config.target_modules}")
print(f"  Dropout: {lora_config.lora_dropout}\n")
print(f"  Memory comparison (7B model):")
print(f"    Full fine-tune: ~60GB (4x A100)")
print(f"    LoRA (16-bit):  ~28GB (1x A100)")
print(f"    QLoRA (4-bit):  ~6GB  (1x T4/L4) ← this!")


> **What just happened?** QLoRA = two breakthroughs combined: (1) load the model in 4-bit NF4 precision (7B model = ~4GB instead of ~14GB), (2) attach LoRA adapters in bfloat16 on top. Only LoRA params are trained in 16-bit; base model stays frozen in 4-bit. 7B model fine-tuning: Full = 60GB (4× A100). QLoRA = 6GB (1× T4, free on Colab). Same quality for most tasks.


### Step 3 · HuggingFace PEFT — Production Fine-Tuning

The complete training loop with SFTTrainer: config, train, save, merge.

**`03_peft_training.py`** · _Block 3 of 3_

PEFT TRAINING — Complete QLoRA fine-tuning loop — pip install transformers peft trl datasets bitsandbytes


In [ ]:
# PEFT TRAINING — Complete QLoRA fine-tuning loop
# pip install transformers peft trl datasets bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
import torch

# ── 1. Quantized model ──
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)

# model = AutoModelForCausalLM.from_pretrained("google/gemma-2b",
#     quantization_config=bnb, device_map="auto")
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")
# model = prepare_model_for_kbit_training(model)

# ── 2. LoRA config ──
lora = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj","v_proj"],
                  lora_dropout=0.05, task_type="CAUSAL_LM")
# model = get_peft_model(model, lora)

# ── 3. Training data ──
train_data = Dataset.from_dict({"text": [
    "<user>What is the refund policy?</user>\n<assistant>Full refund within 7 days.</assistant>",
    "<user>How much is the course?</user>\n<assistant>14,999 rupees with lifetime access.</assistant>",
] * 50})  # 100 examples for demo

# ── 4. Training arguments ──
args = TrainingArguments(
    output_dir="./lora-netsetos",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,    # effective batch = 16
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
)

# ── 5. Train ──
# trainer = SFTTrainer(model=model, args=args, train_dataset=train_data,
#     tokenizer=tokenizer, max_seq_length=512)
# trainer.train()

# ── 6. Save adapter (only 10-50MB!) ──
# model.save_pretrained("./lora-netsetos-adapter")

# ── 7. Merge adapter into base model for deployment ──
# merged = model.merge_and_unload()
# merged.save_pretrained("./netsetos-merged")

print("QLoRA Fine-Tuning Pipeline:\n")
print(f"  1. Load model in 4-bit (BitsAndBytesConfig)")
print(f"  2. Attach LoRA adapters (LoraConfig + get_peft_model)")
print(f"  3. Prepare dataset in chat template format")
print(f"  4. Train with SFTTrainer (from trl library)")
print(f"  5. Save adapter (~20MB, not the full model!)")
print(f"  6. Merge adapter + base for deployment\n")
print(f"  Adapter size: ~20MB (vs 14GB full model)")
print(f"  Training time: ~30 min on T4 for 500 examples")
print(f"  Cost: free on Colab, ~$2 on GCP")


> **What just happened?** The production hyperparameter recipe: r=16, α=32, all-linear, LR=2e-4, cosine scheduler. The most critical insight from 2025 research: LoRA's optimal LR is ~10× higher than for full fine-tuning . If you change rank, you MUST re-tune LR. This single practice eliminates the need for most LoRA variants.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
